In [1]:
import pickle
from torch.utils.data import DataLoader
from data_utils import latent_MH_collate_fn

/home/maximos/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset_path = 'data/latent_datasets/gjt_CA.pickle'

with open(dataset_path, 'rb') as f:
    dataset = pickle.load(f)

In [3]:
print(len(dataset))
print(dataset[0].keys())
print(dataset[0]['latent'].shape)

1857
dict_keys(['harmony_ids', 'attention_mask', 'pianoroll', 'time_signature', 'h_density_complexity', 'latent'])
(512,)


In [4]:
val_loader = DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=latent_MH_collate_fn)

In [5]:
batch = next(iter(val_loader))

In [9]:
print(batch["harmony_ids"].shape)
print(batch["latent"].shape)

torch.Size([4, 80])
torch.Size([4, 512])


In [7]:
from train_utils import make_mixed_batch

In [8]:
mixed_batch_1 = make_mixed_batch(batch, "latent")
mixed_batch_2 = make_mixed_batch(mixed_batch_1, "latent")

In [49]:
harmony_gt = batch["harmony_ids"]
foreign_ids = mixed_batch_1['harmony_ids']

In [50]:
import torch
from GridMLM_tokenizers import CSGridMLMTokenizer

In [51]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [52]:
print(len(tokenizer.vocab))

355


In [53]:
logits = torch.rand((4, 80,len(tokenizer.vocab)))

In [54]:
import torch.nn.functional as F

def compute_unique_logit_activations(harmony_gt, foreign_ids, logits):
    """
    harmony_gt : [B, L] token ids
    foreign_ids: [B, L] token ids
    logits     : [B, D]

    Returns
    -------
    home_unique_logits_activations    : [B]
    foreign_unique_logits_activations : [B]
    """
    
    # B, D = logits.shape

    # Convert logits to probabilities
    probs = F.softmax(logits, dim=2)   # [B, seq_len, D]
    probs_sum = probs.sum(axis=1)
    B, D = probs_sum.shape

    # Create vocabulary masks
    home_mask = torch.zeros(B, D, dtype=torch.bool, device=logits.device)
    foreign_mask = torch.zeros(B, D, dtype=torch.bool, device=logits.device)

    home_mask.scatter_(1, harmony_gt, True)
    foreign_mask.scatter_(1, foreign_ids, True)

    # Unique tokens
    home_unique_mask = home_mask & (~foreign_mask)
    foreign_unique_mask = foreign_mask & (~home_mask)

    # Sum probabilities
    home_unique_sum = (probs_sum * home_unique_mask).sum(dim=1)
    foreign_unique_sum = (probs_sum * foreign_unique_mask).sum(dim=1)

    # Normalize by total probability mass (softmax sums to 1, but kept explicit)
    total_prob = probs_sum.sum(dim=1)

    home_unique_logits_activations = home_unique_sum / total_prob
    foreign_unique_logits_activations = foreign_unique_sum / total_prob

    return home_unique_logits_activations, foreign_unique_logits_activations
# end compute_unique_logit_activations

In [55]:
print(harmony_gt)
print(foreign_ids)

tensor([[  6, 218, 218, 136, 136,   6, 281, 281,  90,  90,   6, 218, 218,  13,
          13,   6, 158, 158, 303, 303,   6, 101, 101, 101, 101,   6, 274, 274,
          71,  71,   6, 216, 216,  13,  13,   6, 158, 158, 303, 303,   6, 101,
         101, 274, 274,   6, 281, 281,  90,  90,   6, 221, 221, 221, 221,   6,
         100, 100,  90,  90,   6, 212, 212, 212, 212,   6,  15,  15, 187, 187,
           6, 169, 169, 168, 168,   6, 304, 304,  93,  93],
        [  6, 101, 101, 101, 101,   6, 247, 247, 247, 247,   6, 100, 100,  42,
          42,   6,  13,  13,  13,  13,   6, 158, 158, 158, 158,   6, 160, 160,
         303, 303,   6, 101, 101,  13,  13,   6, 160, 160, 303, 303,   6, 101,
         101, 101, 101,   6, 305, 305, 100, 100,   6, 246, 246, 246, 246,   6,
         247, 247,  42,  42,   6, 101, 101, 101, 101,   6, 101, 101,  13,  13,
           6, 158, 158, 158, 158,   6, 158, 158, 158, 158],
        [  6,  43,  43,  43,  43,   6,  14,  14,  14,  14,   6, 339, 339, 339,
         33

In [56]:
home_unique_logits_activations, foreign_unique_logits_activations = compute_unique_logit_activations(harmony_gt, foreign_ids, logits)

In [57]:
print(home_unique_logits_activations)
print(foreign_unique_logits_activations)

tensor([0.0506, 0.0115, 0.0312, 0.0113])
tensor([0.0308, 0.0111, 0.0510, 0.0112])


In [58]:
# Create vocabulary masks
B = 4
D = 5
home_mask = torch.zeros(B, D, dtype=torch.bool, device=logits.device)
foreign_mask = torch.zeros(B, D, dtype=torch.bool, device=logits.device)

harmony_gt = torch.tensor([
    [0,1,1,1,2],
    [0,1,1,1,3],
    [0,1,1,1,4],
    [0,1,1,1,2],
])
foreign_ids = torch.tensor([
    [3,1,1,1,0],
    [2,1,1,1,0],
    [2,1,1,1,0],
    [2,1,1,1,0],
])

home_mask.scatter_(1, harmony_gt, True)
foreign_mask.scatter_(1, foreign_ids, True)

# Unique tokens
home_unique_mask = home_mask & (~foreign_mask)
foreign_unique_mask = foreign_mask & (~home_mask)

In [59]:
print(home_unique_mask)
print(foreign_unique_mask)

tensor([[False, False,  True, False, False],
        [False, False, False,  True, False],
        [False, False, False, False,  True],
        [False, False, False, False, False]])
tensor([[False, False, False,  True, False],
        [False, False,  True, False, False],
        [False, False,  True, False, False],
        [False, False, False, False, False]])


In [60]:
logits = torch.rand((4, 5,5))

In [61]:
probs = F.softmax(logits, dim=2)   # [B, seq, D]
probs_sum = logits.sum(axis=1)
B, D = probs_sum.shape

In [62]:
print(probs_sum)

tensor([[2.7371, 2.9592, 3.4882, 2.4100, 2.1799],
        [2.6498, 1.5094, 4.1989, 1.3703, 3.2297],
        [2.5317, 3.2493, 2.5012, 1.6687, 1.8962],
        [1.8521, 2.8759, 3.2817, 1.8185, 3.2590]])


In [63]:
home_unique_logits_activations, foreign_unique_logits_activations = compute_unique_logit_activations(harmony_gt, foreign_ids, logits)

In [64]:
print(home_unique_logits_activations)
print(foreign_unique_logits_activations)

tensor([0.2315, 0.1551, 0.1804, 0.0000])
tensor([0.1883, 0.2677, 0.2043, 0.0000])
